# Green-screen → gradient compositor

Drop a green-screen render onto the **Upload** button (or click to pick).
Tweak the gradient, texture, and fade in real time. Toggle **Transparent
background** to export an RGBA PNG with the green keyed out (no gradient).

Run all cells once, then interact with the widget at the bottom.


## 1. Setup

In [1]:
# %pip install numpy pillow scipy ipywidgets
import io
from pathlib import Path

import numpy as np
from PIL import Image
from scipy.ndimage import gaussian_filter

import ipywidgets as widgets
from IPython.display import display


## 2. Pipeline

Three steps:

1. **Chroma key + despill** — soft alpha from `greenness = G − max(R, B)`,
   plus G clamped to `max(R, B)` to kill green tint on transparent edges.
2. **Vertical gradient** — linear blend between two RGB endpoints.
3. **Floor texture** — multi-octave Gaussian noise faded in toward the bottom.

The expensive part (Gaussian filter at large sigma) is **cached by image
size + noise params**, so slider changes after the first render are fast.


In [2]:
def chroma_key(rgba, key_lo=20.0, key_hi=150.0):
    """Return (foreground_rgb, alpha). Despill clamps G -> max(R,B)."""
    R, G, B = rgba[..., 0], rgba[..., 1], rgba[..., 2]
    max_rb = np.maximum(R, B)
    greenness = G - max_rb
    alpha = 1.0 - np.clip((greenness - key_lo) / (key_hi - key_lo), 0.0, 1.0)
    G_despilled = np.where(greenness > 0, max_rb, G)
    return np.stack([R, G_despilled, B], axis=-1), alpha


def vertical_gradient(h, w, top_rgb, bot_rgb):
    t = np.linspace(0.0, 1.0, h, dtype=np.float32)[:, None]
    top = np.array(top_rgb, dtype=np.float32)
    bot = np.array(bot_rgb, dtype=np.float32)
    col = top * (1 - t) + bot * t
    return np.broadcast_to(col[:, None, :], (h, w, 3)).copy()


# --- Floor texture, with the noise field cached -------------------------
_noise_cache = {}

def _noise_field(h, w, octaves, horizontal_smear, seed):
    """Expensive bit (Gaussian-blurred white noise). Cached by params."""
    key = (h, w, octaves, horizontal_smear, seed)
    if key in _noise_cache:
        return _noise_cache[key]
    rng = np.random.default_rng(seed)
    tex = np.zeros((h, w), dtype=np.float32)
    for sigma, weight in octaves:
        layer = rng.standard_normal((h, w)).astype(np.float32)
        layer = gaussian_filter(layer, sigma=sigma)
        layer = (layer - layer.mean()) / (layer.std() + 1e-8)
        tex += layer * weight
    tex /= np.abs(tex).max() + 1e-8
    tex = gaussian_filter(tex, sigma=(0.0, horizontal_smear))
    _noise_cache[key] = tex
    return tex


def floor_texture(h, w, fade_start=0.45, fade_curve=1.4,
                  octaves=((60.0, 0.55), (18.0, 0.30), (4.0, 0.15)),
                  horizontal_smear=1.5, seed=7):
    tex = _noise_field(h, w, octaves, horizontal_smear, seed)
    t = np.linspace(0.0, 1.0, h, dtype=np.float32)[:, None]
    fade = np.clip((t - fade_start) / max(1e-6, 1.0 - fade_start), 0, 1) ** fade_curve
    fade = np.broadcast_to(fade, (h, w))
    return (tex * fade)[..., None]


def composite_array(rgba, top_rgb=(0, 0, 0), bot_rgb=(44, 44, 44),
                    tex_strength=7.0, tex_fade_start=0.45,
                    key_lo=20.0, key_hi=150.0, transparent=False):
    """
    rgba : float32 (H, W, 4) — green-screen render.
    transparent : if True, return (H, W, 4) RGBA with despilled foreground
                  and the keyed alpha; gradient + texture are skipped.
                  Otherwise return (H, W, 3) RGB composited on the gradient.
    """
    h, w = rgba.shape[:2]
    fg, alpha = chroma_key(rgba, key_lo=key_lo, key_hi=key_hi)

    if transparent:
        out = np.concatenate([fg, (alpha * 255.0)[..., None]], axis=-1)
        return np.clip(out, 0, 255).astype(np.uint8)

    bg = vertical_gradient(h, w, top_rgb, bot_rgb)
    if tex_strength > 0:
        bg = bg + floor_texture(h, w, fade_start=tex_fade_start) * tex_strength
    a = alpha[..., None]
    out = fg * a + bg * (1.0 - a)
    return np.clip(out, 0, 255).astype(np.uint8)


## 3. UI

Renders at full input resolution. The first render computes and caches the
noise field for that image size; subsequent slider tweaks reuse it and feel
snappy. Sliders use `continuous_update=False`, so re-renders only fire on
mouse release.

When **Transparent background** is on, the gradient and texture controls are
inactive — the export is RGBA with the green keyed out and despilled.


In [3]:
# --- Widgets ----------------------------------------------------------
uploader   = widgets.FileUpload(accept='image/png,image/jpeg',
                                multiple=False,
                                description='Drop / click to upload')
top_pick   = widgets.ColorPicker(value='#000000', description='Top')
bot_pick   = widgets.ColorPicker(value='#2c2c2c', description='Bottom')
tex_slider = widgets.FloatSlider(value=7.0, min=0, max=25, step=0.5,
                                 description='Texture',
                                 continuous_update=False)
fade_slider = widgets.FloatSlider(value=0.45, min=0.0, max=0.95, step=0.05,
                                  description='Fade start',
                                  continuous_update=False,
                                  readout_format='.2f')
transparent_cb = widgets.Checkbox(value=False, description='Transparent background')
preview    = widgets.Image(format='png')
status     = widgets.HTML(value="<i>upload an image to begin</i>")
save_btn   = widgets.Button(description='Save full-res PNG',
                            button_style='primary',
                            icon='save')

bg_controls = (top_pick, bot_pick, tex_slider, fade_slider)

# --- State ------------------------------------------------------------
state = {'rgba': None, 'name': None}

def hex_to_rgb(h):
    h = h.lstrip('#')
    return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))


def current_params():
    return dict(top_rgb=hex_to_rgb(top_pick.value),
                bot_rgb=hex_to_rgb(bot_pick.value),
                tex_strength=tex_slider.value,
                tex_fade_start=fade_slider.value,
                transparent=transparent_cb.value)


def render_preview(*_):
    """Re-composite at full resolution and update the preview widget."""
    if state['rgba'] is None:
        return
    out = composite_array(state['rgba'], **current_params())
    mode = 'RGBA' if out.shape[-1] == 4 else 'RGB'
    buf = io.BytesIO()
    Image.fromarray(out, mode=mode).save(buf, format='PNG')
    preview.value = buf.getvalue()


def on_upload(change):
    if not uploader.value:
        return
    val = uploader.value
    file = val[0] if isinstance(val, tuple) else next(iter(val.values()))
    name = file.get('name') or file.get('metadata', {}).get('name', 'upload.png')
    content = file['content']
    rgba = np.array(Image.open(io.BytesIO(bytes(content))).convert('RGBA'),
                    dtype=np.float32)
    state['rgba'] = rgba
    state['name'] = name
    status.value = (f"<b>Loaded:</b> {name} "
                    f"<span style='opacity:.6'>({rgba.shape[1]}×{rgba.shape[0]})</span>")
    render_preview()


def on_transparent_toggle(change):
    """Disable gradient/texture controls when transparent export is on."""
    for w in bg_controls:
        w.disabled = change['new']
    render_preview()


def on_save(_):
    if state['rgba'] is None:
        status.value = "<span style='color:#c33'>Upload an image first.</span>"
        return
    out = composite_array(state['rgba'], **current_params())
    mode = 'RGBA' if out.shape[-1] == 4 else 'RGB'
    suffix = '_keyed.png' if transparent_cb.value else '_composited.png'
    out_name = Path(state['name']).stem + suffix
    Image.fromarray(out, mode=mode).save(out_name, optimize=True)
    status.value = (f"<b>Saved:</b> <code>{out_name}</code> "
                    f"<span style='opacity:.6'>({out.shape[1]}×{out.shape[0]}, "
                    f"{mode})</span>")


# --- Wire up ----------------------------------------------------------
uploader.observe(on_upload, names='value')
for w in bg_controls:
    w.observe(render_preview, names='value')
transparent_cb.observe(on_transparent_toggle, names='value')
save_btn.on_click(on_save)

display(widgets.VBox([
    uploader,
    widgets.HBox([top_pick, bot_pick]),
    tex_slider,
    fade_slider,
    transparent_cb,
    widgets.HBox([save_btn, status]),
    preview,
]))


## Notes

- **Drag-and-drop** — VS Code's notebook renderer accepts files dropped onto
  the Upload button. Click also works.
- **Full resolution** — preview and save both run at the input resolution.
  The Gaussian-blurred noise field is cached after the first render, so
  later slider tweaks reuse it (only the cheap blend re-runs).
- **Transparent export** — checkbox returns RGBA with the keyed alpha. The
  foreground is despilled (G clamped to max(R,B) where green dominated),
  so transparent edges read as neutral grey instead of sickly green.
- **Filenames** — saves as `<stem>_composited.png` for gradient export,
  `<stem>_keyed.png` for transparent.
- **Determinism** — texture seed is `7`. Change it in `floor_texture` for a
  different noise roll (and clear `_noise_cache` to force a recompute).
